In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 4.9 Relativistic Optics: Doppler, Aberration, and What a Camera Sees

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume IV — Special Relativity",
    number="4.9",
    title="Relativistic Optics: Doppler, Aberration, and What a Camera Sees",
    blurb="The observational coda to the volume: what light actually "
    "delivers to an eye or a telescope. The Doppler shift and aberration "
    "from one boost of a photon's four-momentum, the twin paradox settled "
    "by counting light pulses, the CMB dipole to four digits, jets that "
    "seem to outrun light, and the photograph that shows a flying cube "
    "rotated rather than squashed.",
    difficulty="intermediate",
    estimate="130–160 min",
)

## Notebook overview

Volume IV has transformed coordinates, momenta, and fields; its capstone
even bent light around the Sun. What it has not yet done is ask the
observational question: *what does an observer actually receive?* Every
quantity a telescope measures arrives as light, and light carries two
relativistic imprints at once — its frequency shifts (Doppler) and its
direction swings (aberration). Both fall out of a single Lorentz boost of
the photon's four-momentum, and this coda works out what they do to the
real sky.

The applications are not toy problems. The motion of the solar system
through the cosmic microwave background writes a dipole of exactly
$3.36\ \mathrm{mK}$ onto a $2.7255\ \mathrm{K}$ sky, and we reproduce the
measured Planck value {cite}`planck2020` to four digits from one velocity.
Quasar jets appear to move at six times the speed of light, and the
resolution {cite}`rees1966` is an exercise in light-travel time, not a
crisis. The twin paradox of [§4.4](paradoxes.ipynb) gets its cleanest
settlement here, by Bondi's trick {cite}`bondi1964` of having the twins
literally *watch* each other through exchanged light pulses. And the
finale answers a question Einstein's readers asked for fifty years before
Terrell and Penrose did {cite}`terrell1959,penrose1959`: does a camera
*see* the Lorentz contraction? We build the camera — a light-travel-time
renderer — and photograph a passing cube and sphere to find out. The
volume's standing references remain {cite}`taylor_wheeler,nolting4`.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid convention,
or too tight a tolerance. Treat a ✗ as a prompt to locate the
discrepancy. Passing is strong evidence, not proof.

## Theory in brief

We use natural units $c = 1$ throughout (velocities are $\beta$, times
and distances share units), restoring SI values only where a physical
number is quoted.

**One boost does everything.** A photon of angular frequency $\omega$
travelling at angle $\theta$ to the $x$-axis has four-momentum
$p^\mu = \omega\,(1, \cos\theta, \sin\theta, 0)$ (units $\hbar = 1$; only
the direction matters). An observer moving at $+\beta\hat{\mathbf x}$
relative to the source sees the boosted components, and reading them off
gives both effects at once:

```{math}
:label: eq-ro-doppler
\omega' \;=\; \gamma\,\omega\,(1 - \beta\cos\theta),
```

```{math}
:label: eq-ro-aberration
\cos\theta' \;=\; \frac{\cos\theta - \beta}{1 - \beta\cos\theta}.
```

Equation {eq}`eq-ro-doppler` contains every Doppler case: approach
($\theta = 0$ for a source coming at the observer gives the blueshift
factor $\sqrt{(1+\beta)/(1-\beta)}$ after the sign conventions are
straightened), recession (the reciprocal redshift), and — with no
classical counterpart at all — the *transverse* Doppler shift
$\omega' = \gamma\omega$ at $\theta = \pi/2$, which is time dilation
heard directly. Equation {eq}`eq-ro-aberration` is the swing of
directions: for a fast-moving emitter the whole rear sky folds forward,
and the rest-frame forward hemisphere lands inside the *headlight cone*
of half-angle $\arccos\beta \approx 1/\gamma$. Photon number is
conserved, so beaming concentrates intensity: an isotropic emitter's
photons arrive with the angular density $f(\mu') = (1-\beta^2)\,/\,
\bigl[2\,(1-\beta\mu')^2\bigr]$ per unit $\mu' = \cos\theta'$.

**Bondi's $k$-factor.** For pulses exchanged along the line of motion the
longitudinal Doppler factor

```{math}
:label: eq-ro-kfactor
k(\beta) \;=\; \sqrt{\frac{1+\beta}{1-\beta}}
```

is the ratio of reception to emission intervals, and it obeys two exact
identities that make it a complete substitute for the Lorentz
transformation: $k(\beta)\,k(-\beta) = 1$ (what recedes redshifted
returns blueshifted) and $\gamma = \tfrac12(k + 1/k)$. Bondi's
$k$-calculus {cite}`bondi1964` runs special relativity on these two
identities alone, and its finest hour is the twin paradox: let the twins
*send each other birthday greetings* and count what each receives. The
asymmetry of the ages comes out of the bookkeeping with nothing hidden.

**The CMB dipole.** An observer moving at $\beta$ through a thermal
radiation bath of temperature $T_0$ sees, in direction $\theta$ from the
motion, a perfect blackbody of temperature

```{math}
:label: eq-ro-dipole
T(\theta) \;=\; \frac{T_0}{\gamma\,\bigl(1 - \beta\cos\theta\bigr)},
```

because the Doppler factor rescales every photon frequency *and* the
Planck spectrum is form-invariant under such a rescaling. To first order
in $\beta$ this is $T_0(1 + \beta\cos\theta)$: a dipole. The solar
system moves at $369.82\ \mathrm{km\,s^{-1}}$ against the CMB frame
{cite}`planck2020`, so the $2.7255\ \mathrm{K}$ sky should carry a
$T_0\beta = 3.362\ \mathrm{mK}$ dipole plus a kinematic quadrupole
smaller by another factor $\sim\beta$: a prediction we check against the
measured sky to four digits.

**Superluminal motion.** A blob launched from a quasar core at speed
$\beta$ at angle $\theta$ to the line of sight chases its own light. In
an observation interval it moves closer, so its images arrive
time-compressed, and the apparent transverse speed on the sky is

```{math}
:label: eq-ro-betaapp
\beta_{\rm app} \;=\; \frac{\beta\sin\theta}{1 - \beta\cos\theta},
```

which exceeds $1$ for fast blobs at small angles: maximizing over
$\theta$ gives $\beta_{\rm app}^{\max} = \gamma\beta$ at
$\cos\theta = \beta$. Observed superluminal speeds therefore put a
*floor* on the true Lorentz factor, $\gamma \ge \sqrt{1 +
\beta_{\rm app}^2}$ — light-travel time turned into a measuring
instrument {cite}`rees1966`.

**The photographed world.** A camera snapshot is *not* a constant-time
slice: light reaching the lens at the moment $t_{\rm obs}$ left different
parts of the object at different earlier times. For a point riding the
uniform worldline $\mathbf r(t) = \mathbf r_0 + \beta t\,\hat{\mathbf x}$
and a camera at $\mathbf r_c$, the emission time is fixed by the backward
light cone,

```{math}
:label: eq-ro-lightcone
\bigl|\mathbf r(t_e) - \mathbf r_c\bigr| \;=\; t_{\rm obs} - t_e,
```

which for uniform motion is a *quadratic* in $t_e$ — the renderer needs
no numerical root-finder at all. Terrell and Penrose discovered what this
implies {cite}`terrell1959,penrose1959`: for a small object at closest
approach, the light-travel delays conspire with the Lorentz contraction
so exactly that the photograph shows the object *rotated* by
$\alpha = \arcsin\beta$, not flattened — and a sphere presents a
perfectly circular outline at any speed, with its *rest* radius. The
contraction is real (it is in the constant-time slice) but invisible to a
camera; forget the contraction in the renderer and the theorem breaks,
which is precisely how our validation will know the physics is right.

## Setup

Natural units $c = 1$; SI constants enter only through quoted
astronomical numbers. Randomness appears only in the aberration ensemble
and the sphere's surface sampling, and is seeded.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.integrate import quad
from scipy.spatial import ConvexHull

from ecp import animate, draw, validate

rng = np.random.default_rng(0)

C_KMS = 299792.458  # speed of light in km/s, for quoted velocities
T_CMB = 2.7255  # CMB monopole temperature in K
V_SUN_KMS = 369.82  # solar-system barycenter speed vs the CMB frame (Planck)


def gamma_of(beta):
    """Lorentz factor 1/sqrt(1 - beta^2).

    The time-dilation and energy factor of every boost in this notebook.

    Parameters
    ----------
    beta : float or numpy.ndarray
        Speed(s) in units of c.

    Returns
    -------
    float or numpy.ndarray
        The Lorentz factor.
    """
    return 1.0 / np.sqrt(1.0 - np.asarray(beta) ** 2)


def boost4(beta):
    """Lorentz boost matrix along +x for four-vectors (t, x, y, z).

    The standard-configuration boost of §4.2, as a 4x4 matrix acting on
    contravariant components: an observer moving at +beta measures these
    primed components.

    Parameters
    ----------
    beta : float
        Boost speed in units of c.

    Returns
    -------
    numpy.ndarray
        The 4x4 boost matrix.
    """
    g = gamma_of(beta)
    L = np.eye(4)
    L[0, 0] = L[1, 1] = g
    L[0, 1] = L[1, 0] = -g * beta
    return L


def photon_p(omega, theta):
    """Four-momentum (omega, omega cos(theta), omega sin(theta), 0) of a photon.

    A null vector: frequency times the lightlike direction in the x-y
    plane, the object whose single boost yields both eq-ro-doppler and
    eq-ro-aberration.

    Parameters
    ----------
    omega : float
        Angular frequency in the source frame.
    theta : float
        Propagation angle from the +x axis, radians.

    Returns
    -------
    numpy.ndarray
        The four-momentum components (t, x, y, z).
    """
    return omega * np.array([1.0, np.cos(theta), np.sin(theta), 0.0])


def k_factor(beta):
    """Bondi's longitudinal Doppler factor sqrt((1+beta)/(1-beta)).

    The ratio of pulse reception to emission intervals along the line of
    motion, eq-ro-kfactor: the entire content of the Lorentz
    transformation for radar-style bookkeeping.

    Parameters
    ----------
    beta : float or numpy.ndarray
        Recession speed in units of c (negative for approach).

    Returns
    -------
    float or numpy.ndarray
        The k-factor.
    """
    b = np.asarray(beta)
    return np.sqrt((1.0 + b) / (1.0 - b))


def photograph(pts_lab, beta, t_obs, cam):
    """Directions from which a camera sees uniformly moving points.

    Implements the backward-light-cone condition eq-ro-lightcone for
    points at lab-frame positions ``pts_lab`` at t = 0, all moving at
    +beta along x. For uniform motion the condition is a quadratic in the
    emission time, solved here in closed form (the root with t_e < t_obs);
    the returned vectors point from the camera to each point's APPARENT
    (retarded) position. This is the whole camera: no lens model needed,
    because only directions matter for a distant snapshot.

    Parameters
    ----------
    pts_lab : numpy.ndarray, shape (N, 3)
        Lab-frame positions of the points at t = 0. For an object with
        rest-frame geometry, the x-extents must already carry the 1/gamma
        Lorentz contraction: the camera photographs the LAB frame.
    beta : float
        Common speed of the points along +x, units of c.
    t_obs : float
        Reception time at the camera.
    cam : numpy.ndarray, shape (3,)
        Camera position.

    Returns
    -------
    numpy.ndarray, shape (N, 3)
        Vectors from the camera to the retarded positions.
    """
    dx0 = pts_lab[:, 0] - cam[0]
    dy = pts_lab[:, 1] - cam[1]
    dz = pts_lab[:, 2] - cam[2]
    # |(dx0 + beta t_e, dy, dz)| = t_obs - t_e  ->  A t_e^2 + B t_e + C = 0
    A = beta**2 - 1.0
    B = 2.0 * dx0 * beta + 2.0 * t_obs
    C = dx0**2 + dy**2 + dz**2 - t_obs**2
    t_e = (-B + np.sqrt(B * B - 4.0 * A * C)) / (2.0 * A)
    apparent = np.stack(
        [pts_lab[:, 0] + beta * t_e, pts_lab[:, 1], pts_lab[:, 2]], axis=1
    )
    return apparent - cam


def image_plane(dirs, distance):
    """Project view directions onto a distant camera's image plane.

    Scales each direction's transverse components by the camera distance
    (the small-angle pinhole projection), so image coordinates carry the
    same units as the scene and a far camera reproduces transverse
    geometry at unit magnification.

    Parameters
    ----------
    dirs : numpy.ndarray, shape (N, 3)
        Vectors from the camera toward the scene (negative z toward it).
    distance : float
        Camera distance used as the projection scale.

    Returns
    -------
    tuple of numpy.ndarray
        Image-plane coordinates (x_img, y_img).
    """
    return (
        dirs[:, 0] / (-dirs[:, 2]) * distance,
        dirs[:, 1] / (-dirs[:, 2]) * distance,
    )


CUBE_EDGES = [
    (a, b) for a in range(8) for b in range(a + 1, 8) if bin(a ^ b).count("1") == 1
]


def cube_vertices(side):
    """The eight vertices of an axis-aligned cube centered at the origin.

    Vertex i has coordinates read off i's binary digits, so CUBE_EDGES
    (index pairs differing in one bit) are exactly the twelve edges.

    Parameters
    ----------
    side : float
        Edge length.

    Returns
    -------
    numpy.ndarray, shape (8, 3)
        The vertex coordinates.
    """
    h = side / 2.0
    return np.array(
        [
            [(-h, h)[(i >> 2) & 1], (-h, h)[(i >> 1) & 1], (-h, h)[i & 1]]
            for i in range(8)
        ]
    )

## Exercise 1 — Doppler from one boost, in every regime

Everything downstream rests on {eq}`eq-ro-doppler` being exactly what a
Lorentz boost does to a photon, so we verify that first — not by
rederiving the formula but by *colliding* it with the matrix machinery of
[§4.2](lorentz-transformation.ipynb).

**Part a)** Build the photon four-momentum $p^\mu = \omega(1, \cos\theta,
\sin\theta, 0)$ with `photon_p` for $\omega = 1$ on a grid of $181$
angles $\theta \in [0, \pi]$, apply the matrix `boost4(0.6)` to each with
`@`, and verify: (i) every boosted vector is still null,
$E'^2 - |\mathbf p'|^2 = 0$ to `atol=1e-12`; (ii) the boosted energy
matches $\gamma\omega(1 - \beta\cos\theta)$ of {eq}`eq-ro-doppler` to
`rtol=1e-12` at every angle; (iii) the boosted direction matches
{eq}`eq-ro-aberration` to `atol=1e-12`.

**Part b)** Verify the three named regimes at $\beta = 0.6$: the
head-on blueshift $\omega'/\omega = \sqrt{(1+\beta)/(1-\beta)} = 2$
exactly (the photon at $\theta = \pi$, meeting the observer), the
recession redshift $1/2$ ($\theta = 0$), and the transverse shift
$\omega'/\omega = \gamma = 1.25$ at $\theta = \pi/2$ — time dilation with
no classical analogue (`rtol=1e-12` each).

**Part c)** The classical limit. The classical wave Doppler for a
receding source is $\omega'_{\rm cl}/\omega = 1/(1 + \beta)$, and
dividing the relativistic $\sqrt{(1-\beta)/(1+\beta)}$ by it gives
*exactly* $\sqrt{1 - \beta^2} = 1/\gamma$: the entire relativistic
content of longitudinal Doppler is one factor of time dilation on top of
the classical wave-crest counting. Verify the identity
$\omega'_{\rm rel}/\omega'_{\rm cl} = 1/\gamma$ at $\beta = 0.6$ to
`rtol=1e-12`, then verify the limit it implies:
$\bigl[\omega'_{\rm rel}/\omega'_{\rm cl} - 1\bigr]/\beta^2 \to -1/2$
as $\beta \to 0$, evaluated at $\beta = 10^{-2}, 10^{-3}, 10^{-4}$
(`rtol=2e-2` at the smallest): relativity hides beneath two powers of
$\beta$, which is why sound and light Doppler agree in every
non-relativistic laboratory.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    null_resid < 1e-12,
    "a boost maps null vectors to null vectors: the photon stays a photon",
    f"max |E'² - |p'|²| = {null_resid:.1e}",
)
validate.close(
    omega_prime,
    omega_formula,
    "the boosted energy is eq-ro-doppler at all 181 angles: one matrix, "
    "the whole Doppler formula",
    rtol=1e-12,
)
validate.close(
    cos_prime,
    cos_formula,
    "and the boosted direction is eq-ro-aberration at all angles",
    rtol=0.0,
    atol=1e-12,
)
validate.close(
    np.array([blue, red, transverse]),
    np.array([2.0, 0.5, 1.25]),
    "head-on 2, recession 1/2, transverse γ = 1.25: including the shift "
    "with no classical analogue",
    rtol=1e-12,
)
validate.close(
    ratio_identity,
    1.0 / gamma_of(beta_1),
    "relativistic over classical recession Doppler is exactly 1/γ: the "
    "whole relativistic content is one time-dilation factor",
    rtol=1e-12,
)
validate.close(
    ratios_cl[-1],
    -0.5,
    "so the correction to classical Doppler enters at -β²/2: invisible "
    "in every slow laboratory",
    rtol=2e-2,
)

## Exercise 2 — Aberration and the headlight cone

{numref}`fig-ro-headlight-schematic` shows the geometry:
{eq}`eq-ro-aberration` folds a source's rest-frame sky toward its
direction of motion, and a fast emitter shines like a headlight. Two
quantitative signatures pin the effect down. First, the *half-sky
boundary*: the rest-frame forward hemisphere ($\theta \le \pi/2$) maps
exactly onto the lab-frame cone $\theta' \le \arccos\beta$, so precisely
half of all photons land inside that cone (for $\beta \to 1$ its opening
shrinks as $1/\gamma$). Second, the arrival *density*: photon number
conservation under the aberration map gives the lab distribution
$f(\mu') = (1-\beta^2)/\bigl[2(1-\beta\mu')^2\bigr]$ stated in the
theory section (here $\mu' = \cos\theta'$ and the source moves *toward*
$+x$, so the map carries $+\beta$).

In [ ]:
# (solution hidden on the public site)


**Part a)** Draw $2\times10^5$ isotropic rest-frame directions
($\mu = \cos\theta$ uniform on $[-1, 1]$ from the seeded
`numpy.random.default_rng(0)` generator) for a source moving at
$\beta = 0.6$, and map each through the aberration formula for a source
approaching the observer, $\mu' = (\mu + \beta)/(1 + \beta\mu)$. Verify
the half-sky boundary: the fraction with $\mu' > \beta$ (inside the
$\arccos\beta$ cone) equals $1/2$ within $4$ Monte Carlo standard errors
($\sigma = 1/(2\sqrt{N})$).

**Part b)** Verify the arrival density against the exact curve two ways:
(i) the ensemble mean $\langle\mu'\rangle$ against the quadrature value
$\int_{-1}^{1}\mu\,f(\mu)\,d\mu$ (`scipy.integrate.quad`) within $4$
standard errors of the mean; (ii) a $30$-bin `numpy.histogram` of
$\mu'$ (density-normalized) against $f$ evaluated at bin centers, with
maximum relative deviation below $10\%$ (the bound is set by the Monte
Carlo noise of the thinnest rear-sky bins, not by the physics). Plot
histogram and curve
together for $\beta = 0.3, 0.6, 0.9$: the march toward the headlight.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(frac_cone - 0.5) < 4.0 * se_frac,
    "the rest-frame forward hemisphere maps onto the arccos β cone: "
    "half of all photons inside, within 4σ",
    f"{frac_cone:.5f} vs 0.5 ± {4 * se_frac:.5f}",
)
validate.check(
    abs(mean_mc - mean_quad) < 4.0 * se_mean,
    "the ensemble's mean arrival direction meets the quadrature of the "
    "exact density within 4σ of the mean",
    f"{mean_mc:.5f} vs {mean_quad:.5f}",
)
validate.check(
    hist_dev < 0.10,
    "and the full 30-bin arrival histogram traces the Jacobian density "
    "to better than 10% everywhere (the thinnest bins' noise floor)",
    f"max deviation {hist_dev:.3f}",
)

## Exercise 3 — The twin paradox, settled by counting pulses

[§4.4](paradoxes.ipynb) resolved the twin paradox with worldline proper
times. Bondi's $k$-calculus {cite}`bondi1964` resolves it more vividly:
the twins *watch each other* through light pulses, and the asymmetry
appears in what each one actually receives, pulse by pulse.

The scenario, concrete: the traveller departs at $\beta = 0.6$
($\gamma = 1.25$, $k = 2$ by {eq}`eq-ro-kfactor`), coasts out for $4$
years of *proper* time, turns instantaneously, and returns at the same
speed, arriving home $8$ traveller-years old while $10$ home-years have
passed. Each twin emits one greeting per year of their own proper time.
The $k$-factor bookkeeping: during the traveller's $4$ outbound years
they receive home's greetings slowed by $1/k = 1/2$ (so $2$ of them),
and during the $4$ inbound years sped up by $k = 2$ (so $8$): total
$10$, the home twin's full age — received asymmetrically, $2$ then $8$.
The home twin sees the reverse asymmetry *in time*: they receive the
traveller's $8$ greetings as $4$ slow ones spread over the first $8$
years (the turnaround signal takes years to arrive!) and $4$ fast ones
crammed into the last $2$.

**Part a)** Verify the two structural identities of the $k$-calculus at
$\beta = 0.6$ and on a grid of $50$ speeds $\beta \in [0.01, 0.95]$:
$k(\beta)\,k(-\beta) = 1$ and $\gamma = \tfrac12(k + 1/k)$, both to
`rtol=1e-12` (they are algebraic identities; the check is on the
implementation).

**Part b)** Simulate the exchange literally. Home sits at $x = 0$; the
traveller's worldline runs to $x = 3$ (light-years) at $t = 5$ and back,
arriving at $t = 10$. Emit greetings at unit proper-time intervals along
each worldline, propagate each pulse at $\pm 45°$ in the $(x, t)$ plane,
and intersect it with the receiver's (piecewise-linear) worldline in
closed form. Verify the counts: the traveller receives exactly $2$
greetings before turnaround and $8$ after; home receives exactly $4$ in
the first $8$ years and $4$ in the final $2$ (greetings arriving exactly
at reunion count as received). Draw the full exchange on a spacetime
diagram built from `ecp.draw.spacetime_diagram` and `worldline`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    prod_resid < 1e-12 and gam_resid < 1e-12,
    "the k-calculus identities k(β)k(−β) = 1 and γ = (k + 1/k)/2 hold "
    "across 50 speeds: the k-factor carries the whole transformation",
    f"residuals {prod_resid:.1e}, {gam_resid:.1e}",
)
validate.check(
    n_before_turn == 2 and n_after_turn == 8,
    "the traveller literally receives 2 greetings outbound and 8 inbound: "
    "ten home-years watched, asymmetrically",
    f"counts {n_before_turn} + {n_after_turn}",
)
validate.check(
    n_first8 == 4 and n_last2 == 4,
    "home receives 4 slow greetings over 8 years and 4 fast ones in the "
    "final 2: eight traveller-years, the age gap with nothing hidden",
    f"counts {n_first8} + {n_last2}",
)

## Exercise 4 — The CMB dipole, to four digits

The most precisely measured Doppler effect in nature is written across
the entire sky. The solar system moves at $v = 369.82\ \mathrm{km\,
s^{-1}}$ with respect to the frame in which the cosmic microwave
background is isotropic {cite}`planck2020`, so by {eq}`eq-ro-dipole`
every direction of the $T_0 = 2.7255\ \mathrm{K}$ sky is Doppler-warmed
or cooled by its angle to that motion.

**Part a)** With $\beta = v/c = 1.2336\times10^{-3}$, evaluate
{eq}`eq-ro-dipole` and project it onto Legendre polynomials: compute
$a_\ell = \tfrac{2\ell+1}{2}\int_{-1}^{1} T(\mu)\,P_\ell(\mu)\,d\mu$ for
$\ell = 0, 1, 2, 3$ by `scipy.integrate.quad` with
`numpy.polynomial.legendre.Legendre.basis`. Verify the monopole is
$T_0$ to a part in $10^6$ (the $O(\beta^2)$ shift is below that), and
verify the dipole amplitude $a_1$ equals the measured Planck dipole
$3.3621\ \mathrm{mK}$ to `rtol=1e-3` — a four-digit confrontation
between {eq}`eq-ro-dipole` and the sky.

**Part b)** The kinematic quadrupole. Expanding {eq}`eq-ro-dipole` to
second order predicts $a_2/a_1 = 2\beta/3$: a quadrupole suppressed by
one more power of $\beta$, about $8.2\times10^{-4}$ of the dipole (and
a real nuisance term for CMB analysts, who must subtract it before
reading the primordial quadrupole). Verify the projected $a_2/a_1$
matches $2\beta/3$ to `rtol=1e-3`, and that $a_3/a_1 < 10^{-5}$. Plot
$T(\theta) - T_0$ across the sky together with the pure-dipole
approximation; the residual — invisible at plot scale — is the
quadrupole.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    a_ell[0],
    T_CMB,
    "the monopole survives the boost to a part in a million: the mean "
    "sky is barely touched",
    rtol=1e-6,
)
validate.close(
    a_ell[1] * 1e3,
    3.3621,
    "one velocity (369.82 km/s) reproduces the measured Planck dipole "
    "amplitude to four digits",
    rtol=1e-3,
)
validate.close(
    a_ell[2] / a_ell[1],
    2.0 * beta_cmb / 3.0,
    "the kinematic quadrupole rides one power of β below the dipole, at "
    "exactly 2β/3 of it",
    rtol=1e-3,
)
validate.check(
    abs(a_ell[3] / a_ell[1]) < 1e-5,
    "and the octupole is negligible: the pattern is dipole + tiny "
    "quadrupole, nothing more",
    f"a3/a1 = {a_ell[3] / a_ell[1]:.1e}",
)

## Exercise 5 — Jets that outrun their own light

In 1966 — five years *before* the observation — Rees {cite}`rees1966`
predicted that radio astronomers would see quasar jets move across the
sky faster than light, and {numref}`fig-ro-jet-schematic` shows why no
law is broken. A blob launched at speed $\beta$ at angle $\theta$ to the
line of sight moves closer between two emissions, so its images arrive
compressed in time while the transverse displacement is undiminished:
the apparent sky speed is {eq}`eq-ro-betaapp`.

In [ ]:
# (solution hidden on the public site)


**Part a)** Verify the peak. On a grid of $2\times10^5$ angles
$\theta \in (0, \pi)$ at $\beta = 0.95$, maximize {eq}`eq-ro-betaapp`
numerically with `numpy.max` and verify the peak equals $\gamma\beta =
3.0424$ to `rtol=1e-6`, occurring at $\cos\theta = \beta$ (grid argmax
within $10^{-3}$ rad of $\arccos\beta$). Plot the $\beta_{\rm app}(
\theta)$ family for $\beta = 0.9, 0.95, 0.99$ with the superluminal
threshold $\beta_{\rm app} = 1$ marked.

**Part b)** Turn it into an instrument. The M87-like jet component we
take as the worked case moves at an observed $\beta_{\rm app} = 6.0$.
Verify the *minimum* Lorentz factor consistent with that observation is
$\gamma_{\min} = \sqrt{1 + \beta_{\rm app}^2} = 6.083$: scan the
$(\beta, \theta)$ feasibility region on a $4000\times4000$-resolution
grid (for each $\beta$, ask whether any angle achieves
$\beta_{\rm app} \ge 6$) and verify the smallest feasible $\gamma$
matches the closed form to `rtol=1e-3`. An apparent speed, read off a
sequence of radio maps, thus *bounds the true Lorentz factor from
below*: light-travel time as a measuring device.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    peak_num,
    peak_exact,
    "the apparent-speed curve peaks at exactly γβ — apparent speeds up "
    "to 3.04c from a blob at 0.95c",
    rtol=1e-6,
)
validate.check(
    abs(peak_th - np.arccos(beta_jet)) < 1e-3,
    "and the peak sits at cos θ = β, the blob chasing its own light at "
    "the critical angle",
    f"θ_peak = {peak_th:.5f} vs arccos β = {np.arccos(beta_jet):.5f}",
)
validate.close(
    gamma_min_num,
    gamma_min_exact,
    "an observed 6c apparent speed forces γ ≥ √(1+36) = 6.08: the sky "
    "measurement bounds the true Lorentz factor",
    rtol=1e-3,
)

## Exercise 6 — The photograph: Terrell rotation and Penrose's circle

Does a camera see the Lorentz contraction? For half a century after 1905
nearly everyone (Einstein's popularizers included) drew flattened
spheres. Terrell and Penrose {cite}`terrell1959,penrose1959` showed the
drawings were wrong: a photograph records the *retarded* positions of
{eq}`eq-ro-lightcone`, light from the far side having left earlier, and
for a small object at closest approach the delays undo the flattening so
exactly that the image is the rest-frame object *rotated* by
$\alpha = \arcsin\beta$. A sphere, rotationally symmetric, therefore
always photographs as a circle — of its *rest* radius.

The renderer is `photograph`: for uniform motion the light-cone
condition is a quadratic in the emission time, solved vertex by vertex
in closed form (no root-finder), followed by the pinhole projection
`image_plane`. One subtlety carries the entire theorem, so it is worth
stating as a warning: the function takes *lab-frame* positions, so an
object with rest side $L$ must be handed coordinates with the $x$-extent
already contracted to $L/\gamma$. Feed it the uncontracted cube — a
tempting mistake — and the "photograph" comes out visibly sheared
instead of rotated: the theorem is precisely the statement that
*contraction plus retardation equals rotation*, and it needs both
ingredients. **Write this one yourself** — the implementation is the
lesson.

**Part a)** Photograph the cube. Take the cube of rest side $L = 1$
(`cube_vertices`), contract its $x$-coordinates by $1/\gamma$ for
$\beta = 0.6$, and photograph it with the camera at distance $D = 4000$
on the $z$-axis at the closest-approach instant ($t_{\rm obs} = D$, so
the cube's center is imaged at the origin). Verify the Terrell theorem
quantitatively: all eight imaged vertices coincide with the *far-field
projection of the rest cube rigidly rotated by* $\alpha = \arcsin 0.6 =
36.87°$ about the vertical axis, to `atol=1e-3` (the residual is the
finite $L^2/D$ perspective). Then break it on purpose: rerun without the
contraction and verify the mismatch exceeds $0.08$ — the theorem fails
by a visible margin the moment either ingredient is dropped.

A guide to *reading* the resulting image, because the camera sits in
the cube's own plane and the projection takes getting used to: a
$y$-rotated cube seen edge-on shows no top face, so it projects to
nested rectangles — the outer vertical edges are the front face
(imaged width $\cos\alpha = 0.8$), the inner ones the trailing side
face swung into view (width $\sin\alpha = 0.6$). And the broken
render is *not* a different shape in this projection: retardation
without contraction is the pure shear $x \to x - \beta z$, and a
shear projects to the same nested rectangles. What distinguishes them
is the **widths** — the shear leaves the front face at its full width
$1$ and stretches the total to $1 + \beta = 1.6$, where the true
photograph shows $\cos\alpha = 0.8$ and $\cos\alpha + \sin\alpha =
1.4$. Verify exactly that fingerprint: photographed front-face width
$\cos\alpha$ and naive front-face width $1$ (both `rtol=1e-3`), with
total extents $1.4$ versus $1.6$. (The animation of Part c, shot
from $18°$ above the plane, is where the rotation *looks* like one.)

**Part b)** Penrose's circle. Sample $2\times10^4$ points of the unit
rest sphere (the seeded generator's `normal` deviates, each vector
scaled to unit length by `numpy.linalg.norm` — the standard isotropic
sphere sampler), contract to the lab
ellipsoid, and photograph at $\beta = 0.8$. Verify the silhouette is a
circle: the isoperimetric ratio $4\pi A / P^2$ of the imaged points'
convex hull (`scipy.spatial.ConvexHull`) exceeds $0.999$, and the
silhouette's half-extents in $x$ and $y$ both equal the *rest* radius
$1$ within $10^{-2}$: the contracted ellipsoid photographs as the
uncontracted circle.

**Part c)** Animate the fly-by — through a *tracking telescope*. A
fixed image plane is useless here: the turn only unfolds across a wide
sweep of sight angles, so each frame re-aims the camera at the cube's
imaged center (unit sight vector $\hat n$, transverse basis
$\hat e_1 = \hat n \times \hat y$, $\hat e_2 = \hat e_1 \times \hat n$)
and scales by the instantaneous distance, exactly what a telescope
tracking the object records; the movie itself is shot from $18°$
above the cube's plane so its top face shows, while every measured
number below comes from the in-plane view Part a certified. Render $180$ frames across sight angles
$\theta \in [-68.8°, +68.8°]$ and measure each frame's apparent
rotation $\varphi$ from the imaged body axes (`numpy.arctan2` of the
two edge extents). Verify the turn: $\varphi$ increases
*monotonically* from $-48.9°$ (approaching: counter-rotated, front
face angled toward the camera) through $0°$ — a moment when the
relativistic photograph is *exactly face-on* near sight angle
$-37°$ — through $\arcsin\beta = 36.87°$ at closest approach
(`atol=0.1°`, re-certifying Part a in the tracking frame) to
$+79.3°$ receding, the rear face swung well into view: a total turn
of $128°$, and never a flattening.

```{admonition} With your assistant
:class: tip
The wireframe-drawing loop (project each vertex, connect the twelve
`CUBE_EDGES`, update per frame) is plotting boilerplate your assistant
can write from this description. The check is yours, and it is the
theorem itself: rerun Part b)'s isoperimetric test at $\beta = 0.95$
and confirm the silhouette is still a circle of rest radius one. If
your renderer shows an ellipse, the contraction or the retardation is
missing. The circle is the physics.
```

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    terrell_err < 1e-3,
    "the closest-approach photograph of the contracted cube IS the rest "
    "cube rotated by arcsin β = 36.87°, to the far-field residual",
    f"max mismatch {terrell_err:.1e}",
)
validate.check(
    naive_err > 0.08,
    "and dropping the contraction breaks the theorem by a visible "
    "margin: rotation = contraction + retardation, both required",
    f"naive mismatch {naive_err:.3f}",
)
validate.close(
    np.array([w_front_photo, w_total_photo]),
    np.array([np.cos(alpha_T), np.cos(alpha_T) + np.sin(alpha_T)]),
    "the widths are the fingerprint of a ROTATION: front face cos α, "
    "total cos α + sin α",
    rtol=1e-3,
)
validate.close(
    np.array([w_front_naive, w_total_naive]),
    np.array([1.0, 1.0 + beta_T]),
    "while retardation without contraction is the pure SHEAR "
    "x → x − βz: front face still 1, total stretched to 1 + β",
    rtol=1e-3,
)
validate.check(
    iso_ratio > 0.999,
    "the photographed sphere silhouette is a circle (isoperimetric ratio "
    "within 1e-3 of 1) at β = 0.8",
    f"4πA/P² = {iso_ratio:.6f}",
)
validate.close(
    np.array([half_x, half_y]),
    np.array([1.0, 1.0]),
    "and its radius is the REST radius: the contraction is real but "
    "cannot be photographed",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([phi_mid]),
    np.array([np.degrees(np.arcsin(beta_T))]),
    "the tracking telescope's closest-approach frame re-certifies "
    "Part a: apparent rotation arcsin β to a tenth of a degree",
    rtol=0.0,
    atol=0.1,
)
validate.check(
    bool(np.all(np.diff(phi_seq) > 0.0)),
    "and the turn is MONOTONE across the whole fly-by: the cube "
    "rotates continuously, frame by frame — it never flattens",
    f"φ spans {phi_seq[0]:+.1f}° → {phi_seq[-1]:+.1f}°",
)
validate.check(
    phi_seq[0] < -45.0
    and phi_seq[-1] > 75.0
    and float(np.min(np.abs(phi_seq[: N_FRAMES_RO // 2]))) < 1.0,
    "the sweep runs from counter-rotated approach through an exactly "
    "face-on moment to the rear face in view: a 128-degree turn, "
    "delivered entirely by light-travel time",
    f"φ: {phi_seq[0]:+.1f}° → 0 (face-on) → {phi_seq[-1]:+.1f}°",
)

## Notebook summary

- One boost of the photon four-momentum delivered both
  {eq}`eq-ro-doppler` and {eq}`eq-ro-aberration` machine-exactly at all
  $181$ test angles, with the three named regimes (blueshift $2$,
  redshift $1/2$, transverse $\gamma = 1.25$ at $\beta = 0.6$) and the
  classical limit recovered at order $\beta^2/2$.
- Aberration folded exactly half of an isotropic source's photons into
  the $\arccos\beta$ headlight cone, and the $2\times10^5$-photon
  arrival histogram traced the Jacobian density
  $(1-\beta^2)/[2(1-\beta\mu')^2]$ to better than $6\%$ everywhere.
- Bondi's $k$-calculus settled the twin paradox by literal bookkeeping:
  at $\beta = 0.6$ ($k = 2$), the simulated pulse exchange delivered
  $2 + 8$ greetings to the traveller and $4 + 4$ to home — ten years
  watched by one twin, eight by the other, asymmetry and all.
- The solar system's $369.82\ \mathrm{km\,s^{-1}}$ reproduced the
  measured Planck CMB dipole $3.3621\ \mathrm{mK}$ to four digits, with
  the kinematic quadrupole at exactly $2\beta/3$ of the dipole.
- Apparent superluminal motion peaked at exactly $\gamma\beta$ (at
  $\cos\theta = \beta$), and an observed $6c$ forced
  $\gamma \ge 6.083$: light-travel time as a lower bound on a quasar
  jet's Lorentz factor.
- The light-cone camera photographed the moving cube as the rest cube
  rotated by $\arcsin\beta$ (mismatch $6\times10^{-5}$; deliberately
  dropping the contraction broke it by $0.1$), the tracking-telescope
  fly-by turned it monotonically through $128°$ — counter-rotated on
  approach, exactly face-on once, rear face in view on recession —
  and the sphere's
  silhouette stayed a circle of rest radius $1$ (isoperimetric ratio
  $0.9999$): the Lorentz contraction is real, and no camera will ever
  see it.

## Outlook

- **Relativistic beaming in the wild.** The headlight effect of
  Exercise 2 is why one side of a two-sided quasar jet usually vanishes:
  the approaching side is Doppler-boosted by a large power of the
  Doppler factor while the receding side is dimmed by the same power.
  Combined with Exercise 5's $\gamma_{\min}$, a single radio map
  constrains both speed and angle.
- **The dipole as a cosmology probe.** Subtracting Exercise 4's pattern
  is the first step of every CMB analysis; the residual "intrinsic"
  dipole — whether the matter and radiation frames agree — is an open
  observational question, tested against quasar catalogues.
- **Rendering at $\gamma \gg 1$.** The camera of Exercise 6 extends
  directly to full scenes: add the Doppler shift of {eq}`eq-ro-doppler`
  per pixel for color, the aberration of Exercise 2 for the wide-angle
  warp, and intensity beaming for brightness, and one has the standard
  "relativistic flight" visualizations — the same three formulas, drawn.
- **Accelerated observers.** Every result here assumed uniform motion
  between emission and reception. Along an accelerated worldline the
  $k$-factor becomes time-dependent and horizons appear: the doorway to
  the Rindler wedge and, ultimately, back to the general relativity of
  [§4.8](gr-capstone.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()